# 21.5 Semantic web and RDF

RDF makes knowledge portable by turning every fact into a graph triple that can be matched, joined, and lightly inferred over.

Semantic web work standardizes facts as subject-predicate-object triples. SPARQL-style matching binds variables, joins patterns, and can add subclass-inferred type triples.

Save a copy to Drive to edit.

In [ ]:

import random

import matplotlib.pyplot as plt

random.seed(2105)


def is_var(token):
    return isinstance(token, str) and token.startswith("?")


def bind_pattern(pattern, triple, binding):
    new_binding = dict(binding)
    for want, got in zip(pattern, triple):
        if is_var(want):
            if want in new_binding and new_binding[want] != got:
                return None
            new_binding[want] = got
        elif want != got:
            return None
    return new_binding


def infer_subclass_triples(triples, subclass):
    inferred = set(triples)
    changed = True
    scans = 0
    while changed:
        changed = False
        for subject, predicate, obj in list(inferred):
            scans += 1
            if predicate == "type" and obj in subclass:
                for parent in subclass[obj]:
                    new_triple = (subject, "type", parent)
                    if new_triple not in inferred:
                        inferred.add(new_triple)
                        changed = True
    return inferred, scans


def rdf_match_and_infer(triples, patterns, subclass):
    graph, infer_scans = infer_subclass_triples(triples, subclass)
    bindings = [dict()]
    scans = infer_scans
    for pattern in patterns:
        next_bindings = []
        for binding in bindings:
            for triple in graph:
                scans += 1
                new_binding = bind_pattern(pattern, triple, binding)
                if new_binding is not None:
                    next_bindings.append(new_binding)
        bindings = next_bindings
    return {"graph": graph, "bindings": bindings, "scans": scans}


def make_rdf_ladder():
    return [
        {"name": "D1 Ada Cy Uni", "triples": {("Ada", "type", "Professor"), ("Ada", "worksAt", "Uni"), ("Cy", "type", "Professor"), ("Cy", "worksAt", "Uni")}, "patterns": [("?x", "type", "Professor"), ("?x", "worksAt", "Uni")], "subclass": {"Professor": {"Person"}}, "expected": 2},
        {"name": "D2 more workers", "triples": {("Ada", "type", "Professor"), ("Cy", "type", "Professor"), ("Bo", "type", "Student"), ("Ada", "worksAt", "Uni"), ("Cy", "worksAt", "Uni"), ("Bo", "worksAt", "Lab")}, "patterns": [("?x", "worksAt", "Uni")], "subclass": {"Professor": {"Person"}, "Student": {"Person"}}, "expected": 2},
        {"name": "D3 aliases duplicates", "triples": {("Ada", "type", "Professor"), ("Ada", "type", "Prof"), ("Cy", "type", "Professor"), ("Ada", "worksAt", "Uni"), ("Cy", "worksAt", "Uni")}, "patterns": [("?x", "type", "Person")], "subclass": {"Professor": {"Person"}, "Prof": {"Person"}}, "expected": 2},
        {"name": "D4 catalog graph", "triples": {("Asset1", "type", "Video"), ("Asset1", "ownedBy", "TeamA"), ("Asset2", "type", "Image"), ("Asset2", "ownedBy", "TeamA"), ("Asset3", "type", "Video"), ("Asset3", "ownedBy", "TeamB")}, "patterns": [("?x", "type", "Media"), ("?x", "ownedBy", "TeamA")], "subclass": {"Video": {"Media"}, "Image": {"Media"}}, "expected": 2},
        {"name": "D5 linked data joins", "triples": {("Ada", "type", "Professor"), ("Cy", "type", "AssociateProfessor"), ("Dee", "type", "Researcher"), ("Ada", "worksAt", "Uni"), ("Cy", "worksAt", "Uni"), ("Dee", "worksAt", "Lab"), ("Uni", "locatedIn", "City1"), ("Lab", "locatedIn", "City1"), ("City1", "inCountry", "US")}, "patterns": [("?x", "type", "Person"), ("?x", "worksAt", "?org"), ("?org", "locatedIn", "City1")], "subclass": {"AssociateProfessor": {"Professor"}, "Professor": {"Person"}, "Researcher": {"Person"}}, "expected": 3},
    ]


## The concept, built once (D1)

An RDF triple is $(s,p,o)$. Pattern matching binds variables, and $Professor\sqsubseteq Person$ turns professor type triples into inferred person triples.

In [ ]:

triples = {("Ada", "type", "Professor"), ("Ada", "worksAt", "Uni"), ("Cy", "type", "Professor"), ("Cy", "worksAt", "Uni")}
patterns = [("?x", "type", "Professor"), ("?x", "worksAt", "Uni")]
subclass = {"Professor": {"Person"}}

d1 = rdf_match_and_infer(triples, patterns, subclass)
ada_edges = [triple for triple in triples if triple[0] == "Ada"]
professor_bindings = rdf_match_and_infer(triples, [("?x", "type", "Professor")], subclass)["bindings"]
person_triples = [triple for triple in d1["graph"] if triple[1] == "type" and triple[2] == "Person"]

print("Ada edges", ada_edges)
print("professors", professor_bindings)
print("persons", person_triples)
print("joined", d1["bindings"])


The lesson counts are exact: Ada has $2$ edges, the professor query has $2$ bindings, subclassing infers $2$ person triples, and joining Professor with worksAt Uni keeps $2$ results.

In [ ]:

assert len(ada_edges) == 2
assert len(professor_bindings) == 2
assert len(person_triples) == 2
assert len(d1["bindings"]) == 2


## The dataset ladder

The RDF ladder grows from a hand graph to chained subclass inference and multi-pattern linked-data joins.

In [ ]:

rdf_ladder = make_rdf_ladder()

for rung in rdf_ladder:
    print(rung["name"], "triples", len(rung["triples"]), "patterns", len(rung["patterns"]), "sample", list(rung["triples"])[:3])


## Run the same method across D1-D5

Every rung uses the same inference and pattern-join method. The metric is answer correctness and triples scanned during inference and joins.

In [ ]:

rdf_results = []
for rung in rdf_ladder:
    result = rdf_match_and_infer(rung["triples"], rung["patterns"], rung["subclass"])
    correct = len(result["bindings"]) == rung["expected"]
    rdf_results.append({
        "name": rung["name"],
        "triples": len(rung["triples"]),
        "answers": len(result["bindings"]),
        "scans": result["scans"],
        "correct": correct,
        "result": result,
    })

for row in rdf_results:
    print(row["name"], row["triples"], row["answers"], row["scans"], row["correct"])


## Results visualization

The panels show query bindings. The curve tracks answer count and scan cost as graph size grows.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], rdf_results):
    text = "\n".join(str(binding) for binding in row["result"]["bindings"][:6])
    ax.text(0.03, 0.95, text, va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
ax.plot([row["triples"] for row in rdf_results], [row["scans"] for row in rdf_results], marker="o", label="scans")
ax.plot([row["triples"] for row in rdf_results], [row["answers"] for row in rdf_results], marker="x", label="answers")
ax.set_xlabel("base triples")
ax.set_title("scan cost and answers")
ax.legend()
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: choosing a representation that hides the constraint in opaque strings. The fix stores type, relation, and subclass facts explicitly so joins and inference can inspect them.

In [ ]:

hard = rdf_ladder[-1]
opaque_records = ["Ada Professor Uni City1", "Cy AssociateProfessor Uni City1"]
fixed = rdf_match_and_infer(hard["triples"], hard["patterns"], hard["subclass"])

print("opaque joinable triples", 0)
print("opaque records", opaque_records)
print("fixed answers", fixed["bindings"])


## Evaluate it + Practice

- Metric: query-answer correctness and triples scanned or joined, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add a new subclass ResearchProfessor below Professor and test inferred Person triples.

Practice: Add a duplicate triple and explain why set semantics keep counts stable.

Practice: Write a three-pattern query that joins person, organization, and country.